# Agent Evaluation Runner

Interactive notebook for running and analyzing agent evaluations.

In [ ]:
from pydantic_evals import Dataset

from agents.viz_designer import VisualizationDesigner
from evals.datasets.viz_designer import ALL_CASES, BASIC_CASES

# isort: off
# Handle Jupyter async context
import nest_asyncio

nest_asyncio.apply()  # Required for await in Jupyter
# isort: on

In [3]:
# Prompt Override (experimentation)
EXPERIMENTAL_PROMPT = """
You are a minimalist data visualization expert.
Always prefer the simplest chart type that conveys the information.
"""

# Use default prompt
# agent = VisualizationDesigner()

# Or use experimental prompt
agent = VisualizationDesigner(prompt_override=EXPERIMENTAL_PROMPT)

In [ ]:
# Run Single Case (debugging)
from evals.datasets.viz_designer import TEMPORAL_TREND

result = await agent.arun(TEMPORAL_TREND.inputs)

print(f"Chart: {result.chart_type}")
print(f"Encoding: {result.encoding}")
print(f"Rationale: {result.rationale}")

In [5]:
# Alternative: Sync wrapper (no nest_asyncio needed)
# result = agent.run(TEMPORAL_TREND.inputs)

In [ ]:
# Full Evaluation with Metrics
from evals.metrics import success_rate

dataset = Dataset(cases=BASIC_CASES)
report = await dataset.evaluate(agent.arun, max_concurrency=5)

# Pretty print results
report.print(include_input=True, include_output=True)

# Summary stats
print(f"\nSuccess Rate: {success_rate(report):.1%}")
print(f"Total Cases: {len(BASIC_CASES)}")

In [ ]:
# Safety Rules Check
from evals.evaluators.viz import SafetyRules

safety_dataset = Dataset(cases=ALL_CASES, evaluators=[SafetyRules()])
safety_report = await safety_dataset.evaluate(agent.arun, max_concurrency=5)

print(f"Safety Pass Rate: {success_rate(safety_report):.1%}")